In [7]:
import numpy as np
import pandas as pd

# Task 0
Read the dataset from csv file & perform data cleaning - remove all rows, which contains `?` in some columns.
Also check for data correctness (salary & salary $K).

In [8]:
# Index column is the unnamed first column; "?" marks missing values
df = pd.read_csv(
    "../data/adult.csv", index_col=0, skipinitialspace=True, na_values="?"
)
rows_before = len(df)
df = df.dropna()
print(f"Rows with '?' removed: {rows_before - len(df)}")

# Correctness: ">50K" must match salary K$ > 50, "<=50K" must match <= 50
is_high = df["salary"] == ">50K"
consistent = is_high == (df["salary K$"] > 50)
print(f"Rows with inconsistent salary / salary K$: {(~consistent).sum()}")
df = df.loc[consistent].reset_index(drop=True)
print(f"Rows left: {len(df)}")

Rows with '?' removed: 2399
Rows with inconsistent salary / salary K$: 0
Rows left: 30162


# Task 1
Print the count of men and women in the dataset.

In [9]:
print(df["sex"].value_counts())

sex
Male      20380
Female     9782
Name: count, dtype: int64


# Task 2
Find the average age of men in dataset

In [10]:
print(df.loc[df["sex"] == "Male", "age"].mean())

39.18400392541707


# Task 3
Get the percentage of people from Poland (native-country)

In [11]:
print(f"{(df['native-country'] == 'Poland').mean() * 100:.3f}%")

0.186%


# Task 4
Get the mean and standard deviation of the age for people who earn > 50K per year. After this, get it for those who earn <= 50K.

In [12]:
print(df.groupby("salary")["age"].agg(["mean", "std"]))

            mean        std
salary                     
<=50K   36.60806  13.464631
>50K    43.95911  10.269633


# Task 5
Check, if there are some people without higher education (education: Bachelors, Prof-school, Assoc-acdm, Assoc-voc, Masters, Doctorate), but with > 50K salary

In [13]:
higher_education = [
    "Bachelors", "Prof-school", "Assoc-acdm",
    "Assoc-voc", "Masters", "Doctorate",
]
no_higher_but_rich = (
    ~df["education"].isin(higher_education) & (df["salary"] == ">50K")
)
count = no_higher_but_rich.sum()
print(f"People without higher education and >50K: {count}")
print(df.loc[no_higher_but_rich, "education"].value_counts())

People without higher education and >50K: 3178
education
HS-grad         1617
Some-college    1336
10th              59
11th              59
7th-8th           35
12th              29
9th               25
5th-6th           12
1st-4th            6
Name: count, dtype: int64


# Task 6
Get the statistics of age for each type of education. Use `groupby` and `describe` for this.

In [14]:
print(df.groupby("education")["age"].describe())

               count       mean        std   min   25%   50%   75%   max
education                                                               
10th           820.0  37.897561  16.225795  17.0  23.0  36.0  52.0  90.0
11th          1048.0  32.363550  15.089307  17.0  18.0  28.5  43.0  90.0
12th           377.0  32.013263  14.373710  17.0  19.0  28.0  41.0  79.0
1st-4th        151.0  44.622517  14.929051  19.0  33.0  44.0  56.0  81.0
5th-6th        288.0  41.649306  14.754622  17.0  28.0  41.0  53.0  82.0
7th-8th        557.0  47.631957  15.737479  17.0  34.0  49.0  60.0  90.0
9th            455.0  40.303297  15.335754  17.0  28.0  38.0  53.0  90.0
Assoc-acdm    1008.0  37.286706  10.509755  19.0  29.0  36.0  44.0  90.0
Assoc-voc     1307.0  38.246366  11.181253  19.0  30.0  37.0  45.0  84.0
Bachelors     5044.0  38.641554  11.577566  19.0  29.0  37.0  46.0  90.0
Doctorate      375.0  47.130667  11.471727  24.0  39.0  47.0  54.0  80.0
HS-grad       9840.0  38.640955  13.067730  17.0  2

# Task 7
Compare the married and non-married men salaries. Who earns more? (>50K or <=50K)
Married men are those, whom `marital-status` starts with "Married". Others are not.

In [15]:
men = df.loc[df["sex"] == "Male"]
is_married = men["marital-status"].str.startswith("Married").rename("married")
share = pd.crosstab(is_married, men["salary"], normalize="index")
print(share.round(3))
print(f">50K share: married={share.loc[True, '>50K']:.1%}, "
      f"not married={share.loc[False, '>50K']:.1%}")

salary   <=50K   >50K
married              
False    0.912  0.088
True     0.552  0.448
>50K share: married=44.8%, not married=8.8%


# Task 8
Get the max hours per week some person works. How many people works the same amount of hours per week?

In [16]:
hours = df["hours-per-week"]
max_hours = hours.max()
print(f"Max hours per week: {max_hours}")
print(f"People working that many hours: {(hours == max_hours).sum()}")

Max hours per week: 99
People working that many hours: 78


# Task 9
Analyze the correlation between data in dataset. Understand connected fields in it and print highlight thier connection.

In [17]:
# Encode salary as 0/1 so it can be correlated with numeric columns
numeric = df.assign(high_salary=(df["salary"] == ">50K").astype(int))
corr = numeric.corr(numeric_only=True)
print(corr.round(3))

# Strongest connections between different fields
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
pairs = upper.stack().rename("corr")
print(pairs[pairs.abs() > 0.2].sort_values(key=abs, ascending=False))

                  age  hours-per-week  salary K$  high_salary
age             1.000           0.102      0.208        0.242
hours-per-week  0.102           1.000      0.196        0.229
salary K$       0.208           0.196      1.000        0.854
high_salary     0.242           0.229      0.854        1.000
salary K$       high_salary    0.853894
age             high_salary    0.241998
hours-per-week  high_salary    0.229480
age             salary K$      0.208203
Name: corr, dtype: float64
